# Prototipo de transformación Bronze a Silver con PySpark

Este notebook tiene como objetivo probar las transformaciones necesarias
para construir la capa Silver a partir de las tablas almacenadas en
PostgreSQL Bronze.

En esta primera prueba se trabajará únicamente con "bronze.students"

Objetivo del Notebook:

- Lectura de PostgreSQL mediante JDBC.
- Revisión del esquema de origen.
- Limpieza de espacios.
- Normalización de nombres, correos y países.
- Conversión de fechas.
- Creación de una bandera de calidad temporal.
- Validación de conteos y valores nulos.

Este notebook funciona como prototipo. La lógica definitiva será trasladada
posteriormente a un script dentro de "src/"

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("PrototypeSilverStudents")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.4"
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Conexión con la bd

In [2]:
JDBC_URL = "jdbc:postgresql://postgres:5432/dwh"

JDBC_PROPERTIES = {
    "user": "flor",
    "password": "mundolibre",
    "driver": "org.postgresql.Driver"
}

BRONZE_TABLE = "bronze.students"

In [3]:
df_students_bronze = spark.read.jdbc(
    url=JDBC_URL,
    table=BRONZE_TABLE,
    properties=JDBC_PROPERTIES
)

print(df_students_bronze.count(), "registros")

5000 registros


Esto se utiliza para verificar que todas las columnas deben ser string menos ingested_at

In [4]:
df_students_bronze.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- enrolled_at: string (nullable = true)
 |-- country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _batch_id: string (nullable = true)



# Tranformación a Silver
Creamos un dataframe para no modificar nada en la capa bronze.Se sabe gracias al Discovery que en "Students" hay 5000 filas, 7 columnas, 0 nulos y 0 duplicados

In [5]:
df_students_silver = df_students_bronze.select(
    F.trim(F.col("student_id")).alias("student_id"),

    F.initcap(
        F.trim(F.col("first_name"))
    ).alias("first_name"),

    F.initcap(
        F.trim(F.col("last_name"))
    ).alias("last_name"),

    F.lower(
        F.trim(F.col("email"))
    ).alias("email"),

    F.to_date(
        F.trim(F.col("birth_date"))
    ).alias("birth_date"),

    F.to_timestamp(
        F.trim(F.col("enrolled_at"))
    ).alias("enrolled_at"),

    F.upper(
        F.trim(F.col("country"))
    ).alias("country"),

    F.col("_source_file"),

    F.col("_ingested_at").alias(
        "_bronze_ingested_at"
    ),

    F.col("_batch_id")
)

Con **.trim** se eliminan los espaciones de inicio o fin (ej. si "STU001 " cambia a "STU001"), esto en student_id, first_name, last_name, email, birth_date, enrolled_at, country

Con **.initcap** se pone la primera letra en mayúscula y las que le siguen en minúscula, esto se hace en first_name, last_name

Con **.lower** se convierte todas las letras en minúsculas, esto se hace en email

Con **.upper** se hace la estandarización de los países a mayúsculas (ej. si "Ar" cambia a "AR")

Con **.to_date** se convierte birt_date de string a date (AAAA-MM-DD)

Con **.to_timestamp** se convierte enrolled_at de string a timestamp ("2024-01-10 14:30:00")

Con **.col** en source_file se hace la conservación de la metadata en bronze

In [6]:
df_students_silver = df_students_silver.withColumn(
    "is_temporally_valid",
    F.when(
        F.col("birth_date").isNull() |
        F.col("enrolled_at").isNull(),
        F.lit(False)
    )
    .when(
        F.col("birth_date") <
        F.to_date(F.col("enrolled_at")),
        F.lit(True)
    )
    .otherwise(F.lit(False))
)

Esto indica: True-> Nacimiento anterior a la inscripcion, False-> Fechas nulas o rel temporal mala

In [7]:
df_students_silver = df_students_silver.withColumn(
    "_silver_created_at",
    F.current_timestamp()
)

Se verifica que se hayan aplicado los cambios correspondientes

In [8]:
df_students_silver.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- enrolled_at: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _bronze_ingested_at: timestamp (nullable = true)
 |-- _batch_id: string (nullable = true)
 |-- is_temporally_valid: boolean (nullable = false)
 |-- _silver_created_at: timestamp (nullable = false)



In [9]:
df_students_silver.show(
    n=10,
    truncate=False
)

+-----------+-----------+---------+----------------------------------+----------+-------------------+-------+------------+--------------------------+------------------------------------+-------------------+--------------------------+
|student_id |first_name |last_name|email                             |birth_date|enrolled_at        |country|_source_file|_bronze_ingested_at       |_batch_id                           |is_temporally_valid|_silver_created_at        |
+-----------+-----------+---------+----------------------------------+----------+-------------------+-------+------------+--------------------------+------------------------------------+-------------------+--------------------------+
|STU-0000001|Martina    |Diaz     |martina.diaz5727@lake.local       |2000-12-21|2019-10-01 00:00:00|US     |students.csv|2026-07-17 19:18:05.979767|cc803c97-314a-4a1b-9c5a-87dfca18f5b2|true               |2026-07-18 21:13:18.836736|
|STU-0000002|Manuel     |Torres   |manuel.torres5619@mail.test  

In [10]:
total_bronze = df_students_bronze.count()
total_silver = df_students_silver.count()

print("Registros Bronze:", total_bronze)
print("Registros Silver:", total_silver)

if total_bronze == total_silver:
    print("Validación de conteos: OK")
else:
    print("Validación de conteos: REVISAR")

Registros Bronze: 5000
Registros Silver: 5000
Validación de conteos: OK


In [11]:
duplicados_student_id = (
    df_students_silver
    .groupBy("student_id")
    .count()
    .filter(F.col("count") > 1)
)

cantidad_ids_duplicados = duplicados_student_id.count()

print(
    "Student_id duplicados:",
    cantidad_ids_duplicados
)

duplicados_student_id.show(
    truncate=False
)

cantidad_student_id_nulos = (
    df_students_silver
    .filter(F.col("student_id").isNull())
    .count()
)

print(
    "Student_id nulos:",
    cantidad_student_id_nulos
)

Student_id duplicados: 0
+----------+-----+
|student_id|count|
+----------+-----+
+----------+-----+

Student_id nulos: 0


In [12]:
fechas_invalidas = (
    df_students_silver
    .filter(
        F.col("birth_date").isNull() |
        F.col("enrolled_at").isNull()
    )
)

print(
    "Registros con fechas nulas o no convertibles:",
    fechas_invalidas.count()
)


Registros con fechas nulas o no convertibles: 0


In [13]:
df_students_silver.groupBy(
    "is_temporally_valid"
).count().show()

+-------------------+-----+
|is_temporally_valid|count|
+-------------------+-----+
|               true| 5000|
+-------------------+-----+



In [14]:
df_students_silver.filter(
    F.col("is_temporally_valid") == False
).show(
    n=10,
    truncate=False
)

+----------+----------+---------+-----+----------+-----------+-------+------------+-------------------+---------+-------------------+------------------+
|student_id|first_name|last_name|email|birth_date|enrolled_at|country|_source_file|_bronze_ingested_at|_batch_id|is_temporally_valid|_silver_created_at|
+----------+----------+---------+-----+----------+-----------+-------+------------+-------------------+---------+-------------------+------------------+
+----------+----------+---------+-----+----------+-----------+-------+------------+-------------------+---------+-------------------+------------------+



In [15]:
resumen_validacion = [
    {
        "validacion": "Conteo Bronze vs Silver",
        "resultado": (
            "OK"
            if total_bronze == total_silver
            else "REVISAR"
        ),
        "cantidad": total_silver
    },
    {
        "validacion": "Student ID duplicados",
        "resultado": (
            "OK"
            if cantidad_ids_duplicados == 0
            else "REVISAR"
        ),
        "cantidad": cantidad_ids_duplicados
    },
    {
        "validacion": "Student ID nulos",
        "resultado": (
            "OK"
            if cantidad_student_id_nulos == 0
            else "REVISAR"
        ),
        "cantidad": cantidad_student_id_nulos
    }
]

df_resumen_validacion = spark.createDataFrame(
    resumen_validacion
)

df_resumen_validacion.show(
    truncate=False
)

+--------+---------+-----------------------+
|cantidad|resultado|validacion             |
+--------+---------+-----------------------+
|5000    |OK       |Conteo Bronze vs Silver|
|0       |OK       |Student ID duplicados  |
|0       |OK       |Student ID nulos       |
+--------+---------+-----------------------+



In [16]:
(
    df_students_bronze
    .select(
        F.upper(
            F.trim(F.col("country"))
        ).alias("country")
    )
    .groupBy("country")
    .count()
    .orderBy("country")
    .show(100, truncate=False)
)

+-------+-----+
|country|count|
+-------+-----+
|AR     |523  |
|BR     |427  |
|CL     |1980 |
|CO     |389  |
|ES     |390  |
|MX     |483  |
|PE     |513  |
|US     |295  |
+-------+-----+



Se realizo esta verificación que no existan valores "diferentes pero iguales" (ej. Argentina -> AR) pero como no existen nombres completos no se realizará nada